# 03_01 — Mapa cartográfico SER

## 0. Configuración inicial

Se detecta la raíz del repositorio mediante `data_catalog.csv`, se definen rutas relativas de entrada y salida, y se fijan las constantes cartográficas del notebook. Las operaciones espaciales se trabajan en ETRS89 / UTM zona 30N (`EPSG:25830`) para que los buffers se expresen en metros.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

try:
    import folium
    from folium.plugins import MarkerCluster, Search
except ImportError as exc:
    raise ImportError(
        "Este notebook requiere Folium para construir el mapa interactivo HTML. "
        "No instala paquetes; ejecutarlo en el entorno del proyecto."
    ) from exc


pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError(
        "No se ha encontrado data_catalog.csv al recorrer la ruta actual y sus padres."
    )


def relpath(path: Path) -> str:
    try:
        return path.resolve().relative_to(ROOT).as_posix()
    except ValueError:
        return path.as_posix()


ROOT = find_repo_root()
CATALOG_PATH = ROOT / "data_catalog.csv"

TARGET_CRS = "EPSG:25830"
VISUAL_BUFFER_M = 25
WEB_SIMPLIFY_M = 0.5

INPUT_PATHS = {
    "ser_geoportal_limite_ser": ROOT / "data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_clean.parquet",
    "ser_geoportal_barrios_ser": ROOT / "data/interim/cartografia/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser_clean.parquet",
    "ser_geoportal_bandas_aparcamiento": ROOT / "data/interim/cartografia/ser_geoportal_bandas_aparcamiento/ser_geoportal_bandas_aparcamiento_clean.parquet",
    "callejero_viales_vigentes": ROOT / "data/interim/cartografia/callejero_viales_vigentes/callejero_viales_vigentes_clean.parquet",
    "ser_parquimetros": ROOT / "data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet",
}

MAPS_DIR = ROOT / "reports" / "maps"
FIGURES_DIR = ROOT / "reports" / "figures" / "ser_cartografia"
HTML_OUTPUT = MAPS_DIR / "mapa_ser_cartografia.html"
PNG_OUTPUT = ROOT / "reports/figures/ser_cartografia/mapa_ser_cartografia.png"
PNG_BARRIOS_OUTPUT = ROOT / "reports/figures/ser_cartografia/mapa_ser_cartografia_barrios.png"
PNG_ZOOM_OUTPUT = ROOT / "reports/figures/ser_cartografia/mapa_ser_cartografia_zoom_pradolongo.png"

MAPS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT detectada mediante data_catalog.csv")
print(f"TARGET_CRS = {TARGET_CRS}")
print(f"VISUAL_BUFFER_M = {VISUAL_BUFFER_M}")
print(f"WEB_SIMPLIFY_M = {WEB_SIMPLIFY_M}")

ROOT detectada mediante data_catalog.csv
TARGET_CRS = EPSG:25830
VISUAL_BUFFER_M = 25
WEB_SIMPLIFY_M = 0.5


## 1. Objetivo y uso del mapa en el TFM

Este notebook construye una visualización cartográfica integrada del Servicio de Estacionamiento Regulado (SER) de Madrid a partir de capas limpias ya generadas en `data/interim/`.

El mapa sirve para delimitar el espacio de análisis, situar calles, bandas reguladas y parquímetros, y producir una salida visual utilizable tanto en la memoria del TFM como en la inspección interactiva de las capas. La salida HTML facilita revisar relaciones espaciales de forma interactiva, mientras que las salidas PNG aportan figuras estáticas y reproducibles para documentación: una vista general del ámbito SER, una vista territorial de barrios y un zoom local de detalle.

El notebook no modela dificultad SER, no limpia fuentes, no genera variables finales, no construye joins definitivos y no escribe datasets `processed`. Las operaciones son de lectura, diagnóstico espacial y visualización no destructiva.

## 2. Capas de entrada y contrato de lectura

Las cinco capas se leen desde salidas limpias previas. El límite SER aporta el contorno y la máscara espacial del ámbito regulado; los barrios SER aportan la división territorial interna; las bandas SER representan las líneas coloreadas del estacionamiento regulado y sus plazas; el callejero aporta un fondo vial poligonal e identificación de calles; y los parquímetros ubican la infraestructura puntual SER.

In [2]:
input_contract = pd.DataFrame(
    [
        {"dataset_id": dataset_id, "ruta_interim": relpath(path)}
        for dataset_id, path in INPUT_PATHS.items()
    ]
)

input_contract

,dataset_id,ruta_interim
0,ser_geoportal_limite_ser,data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_clean.parquet
1,ser_geoportal_barrios_ser,data/interim/cartografia/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser_clean.parquet
2,ser_geoportal_bandas_aparcamiento,data/interim/cartografia/ser_geoportal_bandas_aparcamiento/ser_geoportal_bandas_aparcamiento_clean.parquet
3,callejero_viales_vigentes,data/interim/cartografia/callejero_viales_vigentes/callejero_viales_vigentes_clean.parquet
4,ser_parquimetros,data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet


## 3. Lectura y validación mínima de capas limpias

Se leen las capas limpias generadas en notebooks previos y se comprueba que cumplen las condiciones mínimas para construir el mapa: existencia del archivo, ausencia de capas vacías, CRS válido, geometrías utilizables y columnas mínimas necesarias.

Las cuatro capas cartográficas procedentes de `data/interim/cartografia/` se leen como GeoParquet, porque ya contienen geometría persistida. En cambio, `ser_parquimetros_clean.parquet` se lee como tabla Parquet convencional: esta fuente conserva coordenadas (`gis_x`, `gis_y`, `longitud`, `latitud`), pero no una geometría GeoParquet. Por ello, la geometría puntual de los parquímetros se reconstruye en memoria a partir de `gis_x` y `gis_y`, manteniendo `EPSG:25830`.

Esta operación no modifica la fuente limpia ni escribe una nueva capa en `data/`. Solo crea una representación puntual temporal necesaria para visualizar los parquímetros dentro del mapa SER.

In [3]:
MIN_COLUMNS = {
    "ser_geoportal_limite_ser": {"geometry"},
    "ser_geoportal_barrios_ser": {"cod_barrio", "barrio", "geometry"},
    "ser_geoportal_bandas_aparcamiento": {"id_banda", "color", "numero_plazas", "geometry"},
    "callejero_viales_vigentes": {"top_id", "nombre_via_completo", "geometry"},
    "ser_parquimetros": {"gis_x", "gis_y", "longitud", "latitud"},
}


def _is_missing_geo_metadata_error(exc: Exception) -> bool:
    return "Missing geo metadata" in str(exc)


def _to_numeric_coordinate(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype("string").str.replace(",", ".", regex=False).str.strip(),
        errors="coerce",
    )


def _build_parquimetros_geometry_from_coordinates(df: pd.DataFrame, dataset_id: str) -> gpd.GeoDataFrame:
    required_coordinate_columns = {"gis_x", "gis_y", "longitud", "latitud"}
    missing_coordinate_columns = sorted(required_coordinate_columns - set(df.columns))
    if missing_coordinate_columns:
        raise ValueError(
            f"Dataset: {dataset_id}; el Parquet no tiene metadatos GeoParquet ni columna geometry, "
            f"y faltan columnas de coordenadas: {missing_coordinate_columns}"
        )

    x = _to_numeric_coordinate(df["gis_x"])
    y = _to_numeric_coordinate(df["gis_y"])

    valid_projected = x.notna() & y.notna()
    if not valid_projected.any():
        lon = _to_numeric_coordinate(df["longitud"])
        lat = _to_numeric_coordinate(df["latitud"])
        valid_geographic = lon.notna() & lat.notna()

        if not valid_geographic.any():
            raise ValueError(
                f"Dataset: {dataset_id}; no hay coordenadas válidas ni en gis_x/gis_y "
                "ni en longitud/latitud."
            )

        geometry = gpd.points_from_xy(lon, lat)
        gdf = gpd.GeoDataFrame(df.copy(), geometry=geometry, crs="EPSG:4326")
        return gdf.to_crs(TARGET_CRS)

    geometry = gpd.points_from_xy(x, y)
    gdf = gpd.GeoDataFrame(df.copy(), geometry=geometry, crs=TARGET_CRS)

    if gdf.geometry.isna().all():
        raise ValueError(
            f"Dataset: {dataset_id}; la geometría construida desde gis_x/gis_y queda completamente nula."
        )

    return gdf


def _coerce_geometry_value(value: Any):
    from shapely import wkb, wkt
    from shapely.geometry.base import BaseGeometry

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, BaseGeometry):
        return value

    if isinstance(value, (bytes, bytearray, memoryview)):
        return wkb.loads(bytes(value))

    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None

        try:
            return wkt.loads(text)
        except Exception:
            try:
                return wkb.loads(bytes.fromhex(text))
            except Exception as exc:
                raise ValueError(
                    "No se ha podido interpretar una geometría almacenada como texto."
                ) from exc

    raise TypeError(
        f"Tipo de geometría no soportado en Parquet sin metadatos GeoParquet: {type(value).__name__}"
    )


def read_layer_parquet(dataset_id: str, path: Path) -> gpd.GeoDataFrame:
    try:
        return gpd.read_parquet(path)
    except ValueError as exc:
        if not _is_missing_geo_metadata_error(exc):
            raise ValueError(
                f"Dataset: {dataset_id}; error al leer como GeoParquet: {relpath(path)}"
            ) from exc

        df = pd.read_parquet(path)

        if "geometry" in df.columns:
            geometry = df["geometry"].map(_coerce_geometry_value)
            attributes = df.drop(columns=["geometry"])
            return gpd.GeoDataFrame(attributes, geometry=geometry, crs=TARGET_CRS)

        if dataset_id == "ser_parquimetros":
            return _build_parquimetros_geometry_from_coordinates(df, dataset_id)

        raise ValueError(
            f"Dataset: {dataset_id}; el Parquet no tiene metadatos GeoParquet "
            "y tampoco contiene columna geometry."
        ) from exc


def validate_and_read_layer(dataset_id: str, path: Path) -> gpd.GeoDataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dataset: {dataset_id}; no existe {relpath(path)}")

    gdf = read_layer_parquet(dataset_id, path)

    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(f"Dataset: {dataset_id}; la lectura no devuelve un GeoDataFrame.")
    if gdf.empty:
        raise ValueError(f"Dataset: {dataset_id}; GeoDataFrame vacío.")
    if gdf.crs is None:
        raise ValueError(f"Dataset: {dataset_id}; CRS no declarado.")
    if "geometry" not in gdf.columns:
        raise ValueError(f"Dataset: {dataset_id}; falta columna geometry.")
    if gdf.geometry.isna().all():
        raise ValueError(f"Dataset: {dataset_id}; todas las geometrías son nulas.")

    missing_cols = sorted(MIN_COLUMNS[dataset_id] - set(gdf.columns))
    if missing_cols:
        raise ValueError(f"Dataset: {dataset_id}; faltan columnas mínimas: {missing_cols}")

    try:
        if gdf.crs.to_epsg() != 25830:
            gdf = gdf.to_crs(TARGET_CRS)
    except Exception as exc:
        raise ValueError(f"Dataset: {dataset_id}; CRS no transformable a {TARGET_CRS}.") from exc

    return gdf


layers = {
    dataset_id: validate_and_read_layer(dataset_id, path)
    for dataset_id, path in INPUT_PATHS.items()
}

limite_clean = layers["ser_geoportal_limite_ser"]
barrios_clean = layers["ser_geoportal_barrios_ser"]
bandas_clean = layers["ser_geoportal_bandas_aparcamiento"]
callejero_clean = layers["callejero_viales_vigentes"]
parquimetros_clean = layers["ser_parquimetros"]

validation_check = pd.DataFrame(
    [
        {
            "dataset_id": dataset_id,
            "shape": gdf.shape,
            "crs": gdf.crs.to_string(),
            "n_geometrias_nulas": int(gdf.geometry.isna().sum()),
            "columnas_minimas_ok": sorted(MIN_COLUMNS[dataset_id]),
        }
        for dataset_id, gdf in layers.items()
    ]
)

validation_check

,dataset_id,shape,crs,n_geometrias_nulas,columnas_minimas_ok
0,ser_geoportal_limite_ser,"(1, 3)",EPSG:25830,0,[geometry]
1,ser_geoportal_barrios_ser,"(66, 7)",EPSG:25830,0,"[barrio, cod_barrio, geometry]"
2,ser_geoportal_bandas_aparcamiento,"(34450, 4)",EPSG:25830,0,"[color, geometry, id_banda, numero_plazas]"
3,callejero_viales_vigentes,"(9287, 6)",EPSG:25830,0,"[geometry, nombre_via_completo, top_id]"
4,ser_parquimetros,"(4772, 15)",EPSG:25830,0,"[gis_x, gis_y, latitud, longitud]"


## 4. Diagnóstico espacial previo al mapa

El diagnóstico resume extensión, geometría y cobertura espacial antes de construir vistas cartográficas. El buffer no limpia ni recorta fuentes: se usa únicamente para evaluar cuánto contexto visual conviene conservar alrededor del límite SER.

`VISUAL_BUFFER_M = 25` se toma como criterio inicial conservador. Es suficientemente pequeño para evitar incorporar demasiado contexto urbano, pero permite mantener geometrías viales o puntos situados justo en el borde del polígono SER por precisión cartográfica.

In [4]:
def approximate_bbox(gdf: gpd.GeoDataFrame) -> str:
    minx, miny, maxx, maxy = gdf.total_bounds
    return f"{minx:.0f}, {miny:.0f}, {maxx:.0f}, {maxy:.0f}"


def union_geometry(gdf: gpd.GeoDataFrame):
    if hasattr(gdf.geometry, "union_all"):
        return gdf.geometry.union_all()
    return gdf.geometry.unary_union


layer_summary = pd.DataFrame(
    [
        {
            "dataset_id": dataset_id,
            "shape": gdf.shape,
            "EPSG": gdf.crs.to_epsg(),
            "tipos_geometricos": sorted(gdf.geometry.dropna().geom_type.unique().tolist()),
            "bbox_aprox": approximate_bbox(gdf),
        }
        for dataset_id, gdf in layers.items()
    ]
)

bandas_color_summary = (
    bandas_clean
    .groupby("color", dropna=False)
    .agg(
        n_bandas=("id_banda", "size"),
        plazas=("numero_plazas", "sum"),
    )
    .reset_index()
    .sort_values("color", na_position="last")
)

limite_geom = union_geometry(limite_clean)
buffer_rows = []
for buffer_m in [0, 10, 25, 50]:
    area = limite_geom.buffer(buffer_m)
    buffer_rows.append(
        {
            "buffer_m": buffer_m,
            "n_callejero_intersecta": int(callejero_clean.geometry.intersects(area).fillna(False).sum()),
            "n_parquimetros_intersecta": int(parquimetros_clean.geometry.intersects(area).fillna(False).sum()),
        }
    )

buffer_diagnostic = pd.DataFrame(buffer_rows)

display(layer_summary)
display(bandas_color_summary)
display(buffer_diagnostic)

,dataset_id,shape,EPSG,tipos_geometricos,bbox_aprox
0,ser_geoportal_limite_ser,"(1, 3)",25830,[Polygon],"437012, 4469929, 445598, 4482389"
1,ser_geoportal_barrios_ser,"(66, 7)",25830,[Polygon],"437012, 4469929, 445598, 4482389"
2,ser_geoportal_bandas_aparcamiento,"(34450, 4)",25830,[LineString],"437002, 4469961, 445586, 4482334"
3,callejero_viales_vigentes,"(9287, 6)",25830,"[MultiPolygon, Polygon]","428891, 4462626, 456020, 4499357"
4,ser_parquimetros,"(4772, 15)",25830,[Point],"437010, 4469969, 445594, 4482253"


,color,n_bandas,plazas
0,alta_rotacion,73,372
1,azul,4154,21183
2,naranja,57,1464
3,rojo,32,342
4,verde,30134,157644


,buffer_m,n_callejero_intersecta,n_parquimetros_intersecta
0,0,3266,4761
1,10,3284,4772
2,25,3336,4772
3,50,3411,4772


**Lectura/decisión.** Las capas obligatorias presentan CRS común (`EPSG:25830`) y geometrías compatibles con su función cartográfica: límite y barrios como polígonos, bandas como líneas, callejero como geometría poligonal de viales y parquímetros como puntos reconstruidos en memoria. La extensión del callejero es mayor que la del ámbito SER porque procede de una capa municipal más amplia; por ello no se utiliza completo en la visualización, sino mediante una vista filtrada por intersección con el límite SER ampliado.

La distribución de bandas confirma el predominio de plazas verdes y azules, que serán las categorías visualmente dominantes del mapa. Las categorías de alta rotación, rojo y naranja tienen menor presencia, pero se mantienen para conservar la semántica completa de la capa SER.

El diagnóstico de buffers justifica el uso de `VISUAL_BUFFER_M = 25`: con 10 m ya se recuperan todos los parquímetros, mientras que 25 m añade contexto vial adicional sin incorporar una cantidad excesiva de callejero externo. Este buffer se acepta como criterio visual conservador, no como operación de limpieza ni como transformación de las fuentes.

## 5. Preparación de vistas cartográficas no destructivas

Las vistas para mapa se construyen en memoria. Se conserva completa cada geometría que toca el buffer visual, sin `clip` y sin cortar geometrías. Esto evita partir calles en el borde del SER y mantiene una lectura cartográfica más natural. Si en una iteración posterior la inspección visual muestra demasiado contexto, el ajuste correcto será revisar `VISUAL_BUFFER_M`, no modificar las fuentes limpias.

In [5]:
visual_area = limite_geom.buffer(VISUAL_BUFFER_M)

limite_map = limite_clean
barrios_map = barrios_clean
bandas_map = bandas_clean
callejero_map = callejero_clean.loc[
    callejero_clean.geometry.intersects(visual_area).fillna(False)
].copy()
parquimetros_map = parquimetros_clean.loc[
    parquimetros_clean.geometry.intersects(visual_area).fillna(False)
].copy()

map_views_check = pd.DataFrame(
    [
        {"vista": "limite_map", "shape": limite_map.shape},
        {"vista": "barrios_map", "shape": barrios_map.shape},
        {"vista": "bandas_map", "shape": bandas_map.shape},
        {"vista": "callejero_map", "shape": callejero_map.shape},
        {"vista": "parquimetros_map", "shape": parquimetros_map.shape},
    ]
)

map_views_check

,vista,shape
0,limite_map,"(1, 3)"
1,barrios_map,"(66, 7)"
2,bandas_map,"(34450, 4)"
3,callejero_map,"(3336, 6)"
4,parquimetros_map,"(4772, 15)"


**Lectura/decisión.** Las vistas cartográficas mantienen completo el límite SER, los 66 barrios SER, las 34.450 bandas de aparcamiento y los 4.772 parquímetros. El único subconjunto aplicado afecta al callejero: se conservan 3.336 geometrías que intersectan el ámbito SER ampliado mediante buffer de 25 m. Esta decisión reduce ruido visual y mantiene contexto suficiente en los bordes sin modificar ni recortar la fuente limpia original.

## 6. Construcción del mapa interactivo HTML

El mapa Folium usa `CartoDB Positron` como fondo neutro para facilitar nombres de calles e inspección práctica. Las capas analíticas principales proceden de las geometrías limpias del proyecto y se reproyectan a `EPSG:4326` solo para visualización web.

Para mejorar la interacción, se separan capas visuales y capas consultables. El límite SER y los límites de barrios se muestran como referencias visuales no interactivas, de forma que no bloqueen el clic sobre las bandas. La consulta de barrios se mantiene en una capa separada y desactivada por defecto. Las bandas SER quedan como capa principal de consulta, mientras que los parquímetros se agrupan en `MarkerCluster` para evitar saturación inicial. El callejero propio se mantiene como capa auxiliar y como soporte del buscador de calles mediante `nombre_via_completo`.

In [6]:
COLOR_STYLE = {
    "azul": "#2563eb",
    "verde": "#16a34a",
    "alta_rotacion": "#7c3aed",
    "rojo": "#dc2626",
    "naranja": "#f97316",
}
COLOR_LABEL = {
    "azul": "azul",
    "verde": "verde",
    "alta_rotacion": "alta rotación",
    "rojo": "rojo",
    "naranja": "naranja",
}


def to_web(gdf: gpd.GeoDataFrame, simplify_m: float | None = None) -> gpd.GeoDataFrame:
    web_gdf = gdf.copy()
    if simplify_m is not None and simplify_m > 0:
        web_gdf["geometry"] = web_gdf.geometry.simplify(simplify_m, preserve_topology=True)
    return web_gdf.to_crs("EPSG:4326")


def add_geojson_layer(
    fmap: folium.Map,
    gdf: gpd.GeoDataFrame,
    name: str,
    style_function: Any,
    tooltip_fields: list[str] | None = None,
    tooltip_aliases: list[str] | None = None,
    popup_fields: list[str] | None = None,
    popup_aliases: list[str] | None = None,
    show: bool = True,
    interactive: bool = True,
) -> folium.GeoJson:
    tooltip = None
    if tooltip_fields:
        fields = [field for field in tooltip_fields if field in gdf.columns]
        if fields:
            aliases = tooltip_aliases if tooltip_aliases and len(tooltip_aliases) == len(fields) else None
            tooltip = folium.GeoJsonTooltip(fields=fields, aliases=aliases, sticky=False)
    popup = None
    if popup_fields:
        fields = [field for field in popup_fields if field in gdf.columns]
        if fields:
            aliases = popup_aliases if popup_aliases and len(popup_aliases) == len(fields) else None
            popup = folium.GeoJsonPopup(fields=fields, aliases=aliases, labels=True)

    layer = folium.GeoJson(
        data=gdf.to_json(),
        name=name,
        style_function=style_function,
        tooltip=tooltip,
        popup=popup,
        show=show,
        interactive=interactive,
    )
    layer.add_to(fmap)
    return layer


limite_web = to_web(limite_map, WEB_SIMPLIFY_M)
barrios_web = to_web(barrios_map, WEB_SIMPLIFY_M).reset_index(drop=True)
bandas_web = to_web(bandas_map, WEB_SIMPLIFY_M)
bandas_web["color_label"] = bandas_web["color"].map(COLOR_LABEL).fillna(bandas_web["color"].astype("string"))
bandas_web["tipo_aparcamiento_label"] = "Línea"
callejero_web = to_web(callejero_map, WEB_SIMPLIFY_M)
parquimetros_web = to_web(parquimetros_map)

center = union_geometry(limite_web).centroid
m = folium.Map(
    location=[center.y, center.x],
    zoom_start=13,
    tiles="CartoDB Positron",
    control_scale=True,
)

callejero_layer = add_geojson_layer(
    m,
    callejero_web,
    "Callejero propio/viales vigentes",
    lambda feature: {"color": "#6b7280", "weight": 0.25, "fillColor": "#9ca3af", "fillOpacity": 0.05, "opacity": 0.35},
    tooltip_fields=["top_id", "nombre_via_completo"],
    show=False,
)
Search(
    layer=callejero_layer,
    geom_type="Polygon",
    search_label="nombre_via_completo",
    placeholder="Buscar calle...",
    collapsed=False,
).add_to(m)
add_geojson_layer(
    m,
    barrios_web,
    "Barrios SER — límites",
    lambda feature: {"color": "#374151", "weight": 1.1, "fillOpacity": 0, "opacity": 0.85},
    show=True,
    interactive=False,
)
add_geojson_layer(
    m,
    limite_web,
    "Límite SER",
    lambda feature: {"color": "#000000", "weight": 2.5, "fillOpacity": 0},
    show=True,
    interactive=False,
)
add_geojson_layer(
    m,
    bandas_web,
    "Bandas SER",
    lambda feature: {
        "color": COLOR_STYLE.get(feature["properties"].get("color"), "#4b5563"),
        "weight": 2.4,
        "opacity": 0.9,
    },
    tooltip_fields=["id_banda", "color_label", "tipo_aparcamiento_label", "numero_plazas"],
    tooltip_aliases=["ID", "Color SER", "Tipo aparcamiento", "Número de plazas"],
    popup_fields=["id_banda", "color_label", "tipo_aparcamiento_label", "numero_plazas"],
    popup_aliases=["ID", "Color SER", "Tipo aparcamiento", "Número de plazas"],
    show=True,
)

point_cols = [
    col for col in ["matricula", "cod_barrio", "barrio", "calle", "numero_finca"]
    if col in parquimetros_web.columns
]
parq_group = MarkerCluster(name="Parquímetros SER", show=False)
parq_icon_html = """
    <div style="
        width: 16px; height: 16px; border-radius: 50%;
        background: #2563eb; color: white; border: 1px solid white;
        box-shadow: 0 0 2px rgba(0,0,0,.45);
        font-size: 10px; font-weight: 700; line-height: 16px;
        text-align: center; font-family: Arial, sans-serif;">P</div>
"""
for row in parquimetros_web.itertuples(index=False):
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue
    tooltip = " | ".join(
        str(getattr(row, col)) for col in point_cols if pd.notna(getattr(row, col))
    )
    folium.Marker(
        location=[geom.y, geom.x],
        icon=folium.DivIcon(html=parq_icon_html, icon_size=(16, 16), icon_anchor=(8, 8)),
        tooltip=tooltip or "Parquímetro SER",
    ).add_to(parq_group)
parq_group.add_to(m)

add_geojson_layer(
    m,
    barrios_web,
    "Barrios SER — consulta",
    lambda feature: {"color": "#374151", "weight": 0.8, "fillColor": "#60a5fa", "fillOpacity": 0.08, "opacity": 0.75},
    tooltip_fields=["cod_barrio", "barrio"],
    popup_fields=["cod_barrio", "barrio"],
    show=False,
    interactive=True,
)

bounds = limite_web.total_bounds
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
folium.LayerControl(collapsed=False).add_to(m)
m.save(HTML_OUTPUT)

HTML_OUTPUT

PosixPath('/Users/hugo/TFM_parking_madrid/reports/maps/mapa_ser_cartografia.html')

## 7. Construcción de figuras estáticas para memoria

Se generan tres figuras estáticas: una vista general sintética para memoria, una vista territorial sobria de barrios SER y un zoom de ejemplo en el entorno Pradolongo/Almendrales. No incorporan edificios, manzanas ni portales; priorizan la legibilidad de límite, barrios, viales, bandas y parquímetros según la escala de cada salida.

In [7]:
def set_extent(ax, geom, pad_m: float = 0) -> None:
    minx, miny, maxx, maxy = geom.bounds
    ax.set_xlim(minx - pad_m, maxx + pad_m)
    ax.set_ylim(miny - pad_m, maxy + pad_m)
    ax.set_aspect("equal")
    ax.set_axis_off()


def subset_intersects(gdf: gpd.GeoDataFrame, geom) -> gpd.GeoDataFrame:
    return gdf.loc[gdf.geometry.intersects(geom).fillna(False)].copy()


def plot_bandas(ax, gdf: gpd.GeoDataFrame, linewidth: float, alpha: float, zorder: int) -> None:
    for color, line_color in COLOR_STYLE.items():
        subset = gdf.loc[gdf["color"].eq(color)]
        if subset.empty:
            continue
        subset.plot(ax=ax, color=line_color, linewidth=linewidth, alpha=alpha, label=COLOR_LABEL.get(color, color), zorder=zorder)


def save_figure(fig, path: Path) -> Path:
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path


def label_barrios(ax, gdf: gpd.GeoDataFrame, fontsize: float = 5.2) -> int:
    if "barrio" not in gdf.columns:
        return 0
    labelled = 0
    for row in gdf.itertuples(index=False):
        geom = row.geometry
        name = getattr(row, "barrio")
        if geom is None or geom.is_empty or pd.isna(name):
            continue
        point = geom.representative_point()
        ax.text(
            point.x,
            point.y,
            str(name).title(),
            ha="center",
            va="center",
            fontsize=fontsize,
            color="#111827",
            alpha=0.85,
            zorder=4,
        )
        labelled += 1
    return labelled


fig, ax = plt.subplots(figsize=(12, 12))
callejero_map.plot(ax=ax, color="#f3f4f6", edgecolor="#d1d5db", linewidth=0.12, zorder=1)
barrios_map.boundary.plot(ax=ax, color="#6b7280", linewidth=0.45, alpha=0.65, zorder=2)
plot_bandas(ax, bandas_map, linewidth=0.65, alpha=0.86, zorder=3)
parquimetros_map.plot(ax=ax, color="#1f2937", markersize=1.4, alpha=0.28, label="parquímetros", zorder=4)
limite_map.boundary.plot(ax=ax, color="#000000", linewidth=1.25, label="límite SER", zorder=5)
set_extent(ax, visual_area)
ax.set_title("Mapa cartográfico SER: ámbito, viales, bandas y parquímetros", fontsize=14, pad=16)
ax.legend(loc="lower left", frameon=True, framealpha=0.92, fontsize=8)
save_figure(fig, PNG_OUTPUT)

fig, ax = plt.subplots(figsize=(12, 12))
callejero_map.plot(ax=ax, color="#f8fafc", edgecolor="#d1d5db", linewidth=0.10, alpha=0.85, zorder=1)
barrios_map.boundary.plot(ax=ax, color="#374151", linewidth=0.50, alpha=0.9, zorder=2)
limite_map.boundary.plot(ax=ax, color="#000000", linewidth=1.45, label="límite SER", zorder=3)
n_barrios_etiquetados = label_barrios(ax, barrios_map, fontsize=5.0)
set_extent(ax, visual_area)
ax.set_title("Barrios SER: división territorial del ámbito regulado", fontsize=14, pad=16)
ax.legend(loc="lower left", frameon=True, framealpha=0.92, fontsize=8)
save_figure(fig, PNG_BARRIOS_OUTPUT)

zoom_candidates = barrios_map.loc[
    barrios_map["barrio"].astype("string").str.contains("pradolongo|almendrales", case=False, na=False)
]
if zoom_candidates.empty:
    center_geom = limite_geom.centroid.buffer(650)
else:
    center_geom = union_geometry(zoom_candidates).representative_point().buffer(650)

callejero_zoom = subset_intersects(callejero_map, center_geom)
bandas_zoom = subset_intersects(bandas_map, center_geom)
barrios_zoom = subset_intersects(barrios_map, center_geom)
parquimetros_zoom = subset_intersects(parquimetros_map, center_geom)

fig, ax = plt.subplots(figsize=(10, 10))
callejero_zoom.plot(ax=ax, color="#f3f4f6", edgecolor="#cbd5e1", linewidth=0.35, zorder=1)
barrios_zoom.boundary.plot(ax=ax, color="#6b7280", linewidth=0.75, alpha=0.8, zorder=2)
plot_bandas(ax, bandas_zoom, linewidth=1.35, alpha=0.95, zorder=3)
parquimetros_zoom.plot(
    ax=ax,
    color="#2563eb",
    edgecolor="white",
    linewidth=0.3,
    markersize=22,
    alpha=0.9,
    label="parquímetros",
    zorder=4,
)
limite_map.boundary.plot(ax=ax, color="#000000", linewidth=1.0, alpha=0.8, label="límite SER", zorder=5)
set_extent(ax, center_geom, pad_m=20)
ax.set_title("Zoom SER Pradolongo/Almendrales: bandas, viales y parquímetros", fontsize=13, pad=14)
ax.legend(loc="lower left", frameon=True, framealpha=0.92, fontsize=8)
save_figure(fig, PNG_ZOOM_OUTPUT)

[PNG_OUTPUT, PNG_BARRIOS_OUTPUT, PNG_ZOOM_OUTPUT]

[PosixPath('/Users/hugo/TFM_parking_madrid/reports/figures/ser_cartografia/mapa_ser_cartografia.png'),
 PosixPath('/Users/hugo/TFM_parking_madrid/reports/figures/ser_cartografia/mapa_ser_cartografia_barrios.png'),
 PosixPath('/Users/hugo/TFM_parking_madrid/reports/figures/ser_cartografia/mapa_ser_cartografia_zoom_pradolongo.png')]

## 8. Exportación de salidas cartográficas

Se comprueba la existencia y tamaño de la salida HTML y de las tres figuras PNG generadas por el notebook.

In [8]:
outputs_check = pd.DataFrame(
    [
        {"salida": "html_interactivo", "ruta": relpath(HTML_OUTPUT)},
        {"salida": "png_general_memoria", "ruta": relpath(PNG_OUTPUT)},
        {"salida": "png_barrios", "ruta": relpath(PNG_BARRIOS_OUTPUT)},
        {"salida": "png_zoom_pradolongo", "ruta": relpath(PNG_ZOOM_OUTPUT)},
    ]
)

outputs_check["existe"] = outputs_check["ruta"].map(lambda route: (ROOT / route).exists())
outputs_check["size_mb"] = outputs_check["ruta"].map(
    lambda route: round((ROOT / route).stat().st_size / 1024**2, 3) if (ROOT / route).exists() else pd.NA
)

outputs_check

,salida,ruta,existe,size_mb
0,html_interactivo,reports/maps/mapa_ser_cartografia.html,True,21.656
1,png_general_memoria,reports/figures/ser_cartografia/mapa_ser_cartografia.png,True,5.118
2,png_barrios,reports/figures/ser_cartografia/mapa_ser_cartografia_barrios.png,True,3.256
3,png_zoom_pradolongo,reports/figures/ser_cartografia/mapa_ser_cartografia_zoom_pradolongo.png,True,2.013


## 9. Lectura final y limitaciones

El notebook genera un mapa base SER integrado. El HTML aporta inspección interactiva con base cartográfica neutra, búsqueda de calles sobre `nombre_via_completo`, límite SER y límites de barrios como referencias visuales no interactivas, barrios consultables mediante capa específica, bandas reguladas con atributos básicos y parquímetros agrupados para evitar saturación inicial. El basemap se utiliza únicamente como apoyo visual y no como fuente analítica del TFM.

El PNG general ofrece una vista sintética para memoria. El mapa de barrios prioriza una lectura formal de la división territorial SER mediante contornos, fondo vial ligero y etiquetas de barrio cuando resultan legibles. El zoom de Pradolongo/Almendrales ejemplifica cómo se visualizan en detalle las bandas, la estructura vial y los parquímetros.

Estas salidas son cartográficas y descriptivas, no analíticas ni de modelado. No se genera dificultad SER, no se incorporan nuevas fuentes, no se construyen joins definitivos y las vistas visuales no sustituyen a los datasets limpios: son selecciones temporales y no destructivas para representar mejor el ámbito SER sin escribir nuevas capas en `data/`.